## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-26-02-52-52 +0000,wapo,See how the Army detonated tons of lead explos...,https://www.washingtonpost.com/nation/2026/11/...
1,2026-11-26-02-49-44 +0000,nypost,Fan killed after being shot in the face at NJ ...,https://nypost.com/2026/11/25/us-news/fan-kill...
2,2026-11-26-02-45-27 +0000,nyt,"Veruschka, One of the World’s First Supermodel...",https://www.nytimes.com/2026/11/25/style/verus...
3,2026-11-26-02-40-02 +0000,missionlocal,Kate Sofis resigns post as head of San Francis...,https://missionlocal.org/2026/11/kate-sofis-re...
4,2026-11-26-02-38-47 +0000,latimes,SpaceX gets approval for 1 million data center...,https://www.latimes.com/business/story/2026-11...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2711/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
157,trump,27
15,thanksgiving,22
108,new,20
152,china,17
147,nvidia,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
321,2026-11-25-11-58-00 +0000,wsj,Nvidia’s Jensen Huang is set for at least two ...,https://www.wsj.com/politics/policy/trump-says...,98
261,2026-11-25-15-46-00 +0000,wsj,There was no evidence a Chinese tech executive...,https://www.wsj.com/politics/national-security...,97
222,2026-11-25-17-24-25 +0000,wapo,"Trump proposes new tariffs on China, Europe an...",https://www.washingtonpost.com/politics/2026/1...,94
254,2026-11-25-15-58-36 +0000,nyt,Trump Administration Unveils New Tariffs for C...,https://www.nytimes.com/2026/11/25/business/tr...,92
134,2026-11-25-21-20-13 +0000,wapo,"In backing Nvidia’s Huang, Trump deals his Chi...",https://www.washingtonpost.com/politics/2026/1...,77


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
321,98,2026-11-25-11-58-00 +0000,wsj,Nvidia’s Jensen Huang is set for at least two ...,https://www.wsj.com/politics/policy/trump-says...
301,72,2026-11-25-13-37-56 +0000,nypost,US proposes new tariffs on 14 trading partners...,https://nypost.com/2026/11/25/business/us-prop...
261,62,2026-11-25-15-46-00 +0000,wsj,There was no evidence a Chinese tech executive...,https://www.wsj.com/politics/national-security...
24,61,2026-11-26-01-35-04 +0000,startribune,Bad weather postpones Macy's Thanksgiving Day ...,https://www.startribune.com/macys-announces-th...
82,59,2026-11-25-22-38-54 +0000,nypost,Elon Musk’s plan for SpaceX to launch 1 millio...,https://nypost.com/2026/11/25/business/fcc-app...
308,47,2026-11-25-12-53-36 +0000,nypost,Kilmar Abrego García lands in Spain after self...,https://nypost.com/2026/11/25/world-news/kilma...
296,46,2026-11-25-13-49-04 +0000,cbc,Libyan army says 2 officers were killed in Ukr...,https://www.cbc.ca/news/world/ukraine-russia-l...
272,43,2026-11-25-15-00-22 +0000,cbc,Wall Street recovers most early losses as pull...,https://www.cbc.ca/news/business/stock-market-...
65,41,2026-11-25-23-39-43 +0000,latimes,"Pier collapses, seawalls breached, homes flood...",https://www.latimes.com/california/story/2026-...
399,40,2026-11-25-03-00-00 +0000,wsj,A Democratic push to enact Medicare for All is...,https://www.wsj.com/politics/policy/democrats-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
